In [16]:
import os
import pickle
import pandas as pd
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as spsolve 

In [ ]:
#Files path and configuration
DATA_FILE = '../data/routes.dat'
MODEL_FILE = 'wmf_model.npz'
MAPPINGS_FILE = 'mappings.pkl'

In [ ]:
class CustomWMF:
    def __init__(self, factors=50, regularization=0.1, iterations=15, alpha=15):
        self.factors = factors
        self.regularization = regularization
        self.iterations = iterations
        self.alpha = alpha
        self.user_vectors = None
        self.item_vectors = None
        
    def fit(self, sparse_matrix):
        """Fit the WMF model to the given sparse user-item interaction matrix."""
        num_users, num_items = sparse_matrix.shape
        np.random.seed(42) # Seed for reproducibility
        self.user_vectors = np.random.normal(scale=1./self.factors, size=(num_users, self.factors))
        self.item_vectors = np.random.normal(scale=1./self.factors, size=(num_items, self.factors))
        C = sparse_matrix.copy() * self.alpha
        lambda_I = sparse.eye(self.factors).tocsr() * self.regularization
        
        for i in range(self.iterations):
            print(f"  -> Iteration {i+1}/{self.iterations}...")
            self.user_vectors = self._als_step(C, self.user_vectors, self.item_vectors, lambda_I)
            self.item_vectors = self._als_step(C.T, self.item_vectors, self.user_vectors, lambda_I)
            
    def _als_step(self, C, X, Y, lambda_I):
        """Perform one ALS step to update either user or item vectors."""
        YtY = Y.T.dot(Y)
        for u in range(C.shape[0]):
            row = C.getrow(u)
            if row.nnz == 0: continue
            Y_subset = Y[row.indices]
            A = YtY + Y_subset.T.dot(sparse.diags(row.data).dot(Y_subset)) + lambda_I
            b = Y_subset.T.dot(row.data + 1)
            X[u] = np.linalg.solve(A, b)
        return X

    def recommend_for_new_profile(self, liked_item_indices, N=5):
        """Generate recommendations for a new user profile based on liked item indices."""
        
        Y = self.item_vectors
        YtY = Y.T.dot(Y)
        lambda_I = sparse.eye(self.factors).tocsr() * self.regularization
        confidence_values = np.ones(len(liked_item_indices)) * self.alpha
        Y_subset = Y[liked_item_indices]
        A = YtY + Y_subset.T.dot(sparse.diags(confidence_values).dot(Y_subset)) + lambda_I
        b = Y_subset.T.dot(confidence_values + 1)
        new_user_vector = np.linalg.solve(A, b)
        scores = self.item_vectors.dot(new_user_vector)
        scores[liked_item_indices] = -9999
        best_indices = np.argsort(scores)[::-1][:N]
        return best_indices, scores[best_indices]

In [ ]:
def load_and_prepare_data(filepath):
    """Load the routes data, create a sparse user-item matrix, and generate mappings."""

    col_names = ['Airline', 'Airline ID', 'Source airport', 'Source airport ID', 
                 'Destination airport', 'Destination airport ID', 'Codeshare', 'Stops', 'Equipment']
                 
    df = pd.read_csv(filepath, header=None, names=col_names)
    df = df[['Source airport', 'Destination airport']].dropna()
    
    route_counts = df.groupby(['Source airport', 'Destination airport']).size().reset_index(name='flights_count')
    
    unique_airports = pd.unique(route_counts[['Source airport', 'Destination airport']].values.ravel('K'))
    airport_to_idx = {airport: idx for idx, airport in enumerate(unique_airports)}
    idx_to_airport = {idx: airport for airport, idx in airport_to_idx.items()}
    
    route_counts['source_idx'] = route_counts['Source airport'].map(airport_to_idx)
    route_counts['dest_idx'] = route_counts['Destination airport'].map(airport_to_idx)
    
    sparse_route_matrix = sparse.csr_matrix(
        (route_counts['flights_count'].astype(float), 
        (route_counts['source_idx'], route_counts['dest_idx'])), 
        shape=(len(unique_airports), len(unique_airports))
    )
    
    return sparse_route_matrix, airport_to_idx, idx_to_airport

In [20]:
def get_or_train_custom_model(sparse_matrix, airport_to_idx, idx_to_airport):
    """
    Fulfills the persistence requirement by saving the entire instance of our custom class.
    """
    if os.path.exists(MODEL_FILE) and os.path.exists(MAPPINGS_FILE):
        with open(MODEL_FILE, 'rb') as f:
            model = pickle.load(f)
        with open(MAPPINGS_FILE, 'rb') as f:
            mappings = pickle.load(f)
            airport_to_idx = mappings['airport_to_idx']
            idx_to_airport = mappings['idx_to_airport']
    else:
        # Instantiate our custom class. Iterations set to 15 for faster convergence.
        model = CustomWMF(factors=50, regularization=0.1, iterations=15, alpha=15)
        model.fit(sparse_matrix)
        
        with open(MODEL_FILE, 'wb') as f:
            pickle.dump(model, f)
        with open(MAPPINGS_FILE, 'wb') as f:
            pickle.dump({'airport_to_idx': airport_to_idx, 'idx_to_airport': idx_to_airport}, f)
            
    return model, airport_to_idx, idx_to_airport

In [ ]:
def recommend_new_routes_custom(model, source_airport_iata, sparse_matrix, airport_to_idx, idx_to_airport, N=5):
    """Generate route recommendations for a given source airport using our custom WMF model."""
    if source_airport_iata not in airport_to_idx:
        return f"Error: The airport {source_airport_iata} is not in the database."
    
    airport_idx = airport_to_idx[source_airport_iata]
    
    # Call the recommend method from our custom class.
    # Pass the user's sparse row to filter out current destinations.
    # Pass the airport_idx itself to prevent self-recommendation.
    user_row = sparse_matrix.getrow(airport_idx)
    ids, scores = model.recommend(user_idx=airport_idx, user_sparse_row=user_row, N=N, filter_items=[airport_idx])
    
    recommendations = [(idx_to_airport[idx], score) for idx, score in zip(ids, scores)]
    return recommendations

In [ ]:
#Train the model and generate recommendations for a new user profile based on liked items (airports).
#Small example of a new user profile with liked airports. In a real scenario, this could come from user input or interaction history.

if __name__ == "__main__":
    route_matrix, airport_map, idx_map = load_and_prepare_data(DATA_FILE)
    wmf_model, airport_map, idx_map = get_or_train_custom_model(route_matrix, airport_map, idx_map)
    
    liked_airports_iata = ['LHR', 'CDG', 'BER', 'AMS']
    
    print(f"New User Profile (Liked items): {liked_airports_iata}")
    
    liked_indices = [airport_map[iata] for iata in liked_airports_iata if iata in airport_map]
    
    recs_indices, recs_scores = wmf_model.recommend_for_new_profile(liked_indices, N=5)
    
    print("\nRanking of recommended documents (destinations):")
    for i, (idx, score) in enumerate(zip(recs_indices, recs_scores), 1):
        airport_code = idx_map[idx]
        print(f"{i}. Destination: {airport_code} (Vector Rank Score: {score:.4f})")

  -> Iteration 1/15...
  -> Iteration 2/15...
  -> Iteration 3/15...
  -> Iteration 4/15...
  -> Iteration 5/15...
  -> Iteration 6/15...
  -> Iteration 7/15...
  -> Iteration 8/15...
  -> Iteration 9/15...
  -> Iteration 10/15...
  -> Iteration 11/15...
  -> Iteration 12/15...
  -> Iteration 13/15...
  -> Iteration 14/15...
  -> Iteration 15/15...
New User Profile (Liked items): ['LHR', 'CDG', 'BER', 'AMS']

Ranking of recommended documents (destinations):
1. Destination: FRA (Vector Rank Score: 0.7165)
2. Destination: LGW (Vector Rank Score: 0.5411)
3. Destination: FCO (Vector Rank Score: 0.4712)
4. Destination: BRU (Vector Rank Score: 0.4517)
5. Destination: DUS (Vector Rank Score: 0.3597)
